# zagg paired read — coincident ATL03 × GEDI

Two sensors, one grid. `serc_tdigest_strata.zarr` holds ATL03 signal-stratum
photon digests at **o19** cells; `serc_gedi_flux.zarr` holds GEDI L1B
waveform flux digests at **o18** cells. Both were built on the same o9 SERC
shards, so pairing is pure morton arithmetic: an o12 block covers a 128×128
o19 tensor on the ATL03 side and a 64×64 o18 tensor on the GEDI side, and one
GEDI cell is exactly the 2×2 ATL03 cells beneath it.

Three linked views, each with one set of controls driving both panes:
elevation slices, the 3-D block, and cell-level waveforms.

In [1]:
# %pip install "moczarr>=0.4.0" "zagg[catalog,viz]" ipympl

import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
from moczarr.convention import morton_decimal
from moczarr.hhdc import read_tensors
from moczarr.open import open_leaf
from mortie import generate_morton_children

from zagg.catalog.shardmap import ShardMap
from zagg.config import default_config
from zagg.readers._layout import rowcol_to_rank
from zagg.readers.tdigest_tensor import cell_index, read_cell
from zagg.stats.tdigest import cdf_from_tdigest

# prefer the 'nasa' profile when it exists (laptop); ambient creds otherwise (hub role)
import botocore.session as _bs

if "nasa" in (_bs.Session().full_config.get("profiles") or {}):
    os.environ.setdefault("AWS_PROFILE", "nasa")
OUT = Path("outputs")
STORE = "s3://sliderule-public/zagg-demo"
ATL03 = f"{STORE}/serc_tdigest_strata.zarr"
GEDI = f"{STORE}/serc_gedi_flux.zarr"

timings = {}


class stage:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        timings[self.name] = round(time.perf_counter() - self.t0, 2)
        print(f"[{self.name}] {timings[self.name]:.1f}s")

## Coincidence is free

The two shardmaps carry the **same o9 shard keys** — same AOI, same grid, same
sharding — so a paired read is two `open_leaf` calls on one key. Blocks join
on their o12 morton word; cells join by index arithmetic (`(r, c)` at o18 ↔
`(2r+δr, 2c+δc)` at o19). Ranking below is by *joint* occupancy — GEDI is a
coverage instrument, ICESat-2 a track instrument, so the interesting cells
are where a ground track crosses sampled footprints.

In [2]:
acfg = default_config("atl03_tdigest_healpix_hive")
gcfg = default_config("gedi01b_waveform_healpix_hive")
AFIELD = f"{acfg.output['grid']['child_order']}/h_tdigest_signal"
GFIELD = f"{gcfg.output['grid']['child_order']}/rx_flux"

sm_a = ShardMap.from_json(str(OUT / "shardmap_atl03_serc_o9.json"))
sm_g = ShardMap.from_json(str(OUT / "shardmap_gedi_serc_o9.json"))
common = sorted(set(map(int, sm_a.shard_keys)) & set(map(int, sm_g.shard_keys)))
shard = max(common, key=lambda k: len(sm_g.granules[list(map(int, sm_g.shard_keys)).index(k)]))
print(f"{len(common)} shared o9 shards; reading {shard}")

astore = open_leaf(ATL03, shard)
gstore = open_leaf(GEDI, shard)
with stage("paired tensors: one shard, both sensors"):
    ablocks = {
        b[3]: b
        for b in read_tensors(
            astore, AFIELD, n_bins=64, resolution=1.0, block_order=12, fit="degrade_resolution"
        )
    }
    gblocks = {
        b[3]: b
        for b in read_tensors(
            gstore, GFIELD, n_bins=64, resolution=1.0, block_order=12, fit="degrade_resolution"
        )
    }

pairs = []  # (word, joint mask at o18, atl03 colsums folded 2x2, gedi colsums)
for w, (gt, gm, (goff, gg), _) in gblocks.items():
    if w not in ablocks:
        continue
    at = ablocks[w][0]
    A2 = at.sum(axis=2).reshape(64, 2, 64, 2).sum(axis=(1, 3))  # o19 -> o18 footprint
    G2 = gt.sum(axis=2)
    joint = (A2 > 0) & (G2 > 0)
    if joint.any():
        pairs.append((w, joint, A2, G2))
pairs.sort(key=lambda p: -int(p[1].sum()))
pd.DataFrame(
    [
        {
            "block": morton_decimal(w),
            "joint cells (o18)": int(j.sum()),
            "atl03 photons in joint": int(A2[j].sum()),
            "gedi pe in joint": int(G2[j].sum()),
        }
        for w, j, A2, G2 in pairs[:8]
    ]
)

4 shared o9 shards; reading 5347391238804865033
[paired tensors: one shard, both sensors] 15.0s


,block,joint cells (o18),atl03 photons in joint,gedi pe in joint
0,4331422411132,136,10246,1151326
1,4331422411111,109,7126,841678
2,4331422411113,103,6415,1023847
3,4331422411143,90,6315,545788
4,4331422411423,84,3666,187847
5,4331422411114,78,4118,759413
6,4331422411112,77,1706,730100
7,4331422411311,74,6069,708936


## Paired elevation slices

One control row, two panes. The slider walks ATL03's elevation bins; the GEDI
pane follows to its **nearest bin by absolute elevation** (each block's two
tensors bin from their own data-driven offsets). ATL03 unobserved cells and
empty bins are transparent; ditto GEDI.

In [3]:
from ipywidgets import Checkbox, Dropdown, HBox, IntSlider, VBox, interactive_output

woptions = [
    (f"{morton_decimal(w)}  ({int(j.sum())} joint cells)", i) for i, (w, j, _, _) in enumerate(pairs)
]
_AVMAX = float(max(ablocks[w][0].max() for w, _, _, _ in pairs))
_GVMAX = float(max(gblocks[w][0].max() for w, _, _, _ in pairs))


def paired_slice(pair=0, z=32):
    w, joint, A2, G2 = pairs[pair]
    at, am, (aoff, ag), _ = ablocks[w]
    gt, gm, (goff, gg), _ = gblocks[w]
    elev = aoff + (z + 0.5) * ag
    gz = int(np.clip(round((elev - goff) / gg - 0.5), 0, 63))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))
    sl = at[:, :, z].astype(float)
    sl[(am != 2) | (sl == 0)] = np.nan
    ax1.imshow(sl, origin="lower", cmap="magma", norm=LogNorm(vmin=1, vmax=_AVMAX))
    ax1.set_title(
        f"ATL03 signal — bin {z:02d}: {aoff + z * ag:+.1f} … {aoff + (z + 1) * ag:+.1f} m\n"
        f"{int(np.nansum(sl)):,} photons",
        fontsize=9,
    )
    gl = gt[:, :, gz].astype(float)
    gl[(gm == 0) | (gl == 0)] = np.nan
    ax2.imshow(gl, origin="lower", cmap="viridis", norm=LogNorm(vmin=1, vmax=_GVMAX))
    ax2.set_title(
        f"GEDI flux — bin {gz:02d}: {goff + gz * gg:+.1f} … {goff + (gz + 1) * gg:+.1f} m\n"
        f"{int(np.nansum(gl)):,} photoelectrons",
        fontsize=9,
    )
    for ax, n in ((ax1, 128), (ax2, 64)):
        ax.set_facecolor("#e8e8e8")
        ax.set_xticks([0, n // 2, n - 1])
        ax.set_yticks([0, n // 2, n - 1])
    fig.suptitle(f"block {morton_decimal(w)} — {elev:+.1f} m", fontsize=11)
    plt.tight_layout()
    plt.show()


_pair_dd = Dropdown(options=woptions, value=0, description="block")
_z_sl = IntSlider(min=0, max=63, value=32, description="elev bin")
_out = interactive_output(paired_slice, {"pair": _pair_dd, "z": _z_sl})
display(VBox([HBox([_pair_dd, _z_sl]), _out]))

## The block in 3-D, both sensors

Same block word, real elevations on the vertical axis, so structure lines up
across the panes (drag each to rotate — ipympl backend). By default each pane
colors by its own intensity (photons / photoelectrons, log-scaled — the units
can't share an axis); tick **color by elevation** to put both panes on one
shared colormap and range, so a color reads as the same height in either
sensor.

In [4]:
%matplotlib widget


def _pts(t, off, g, cap=20000):
    xs, ys, zs = np.nonzero(t)
    counts = t[xs, ys, zs].astype("float64")
    if len(xs) > cap:
        keep = np.random.default_rng(0).choice(len(xs), cap, replace=False)
        xs, ys, zs, counts = xs[keep], ys[keep], zs[keep], counts[keep]
    return xs, ys, off + zs * g, counts


def paired_3d(pair=0, by_elev=False, zmode="auto"):
    w, joint, A2, G2 = pairs[pair]
    at, _, (aoff, ag), _ = ablocks[w]
    gt, _, (goff, gg), _ = gblocks[w]
    sets = [
        (_pts(at, aoff, ag), "viridis", "ATL03 signal photons", 128),
        (_pts(gt, goff, gg), "plasma", "GEDI photoelectrons", 64),
    ]
    azz, gzz = sets[0][0][2], sets[1][0][2]
    # elevation mode: ONE colormap and ONE min/max across both sensors, so
    # color means the same height in both panes (two bars, identical range)
    if by_elev:
        shared_norm = plt.Normalize(min(azz.min(), gzz.min()), max(azz.max(), gzz.max()))
    # z-extent pinning: the PLOT axes themselves, not just color -- pin both
    # panes to one sensor's vertical window to compare structure at one scale
    zlim = {"auto": None, "atl03": (azz.min(), azz.max()), "gedi": (gzz.min(), gzz.max())}[zmode]
    fig = plt.figure(figsize=(11, 5.2))
    for k, ((xs, ys, zz, counts), cmap, label, n) in enumerate(sets):
        ax = fig.add_subplot(1, 2, k + 1, projection="3d")
        alpha = np.clip(counts / np.percentile(counts, 98), 0.08, 1.0)
        if by_elev:
            pts = ax.scatter(xs / n, ys / n, zz, c=zz, s=1.5,
                             cmap="viridis", norm=shared_norm, alpha=alpha)
            cb_label = "elevation (m)"
        else:
            pts = ax.scatter(xs / n, ys / n, zz, c=counts, s=1.5,
                             cmap=cmap, norm=LogNorm(), alpha=alpha)
            cb_label = label.split(" ", 1)[1]
        if zlim is not None:
            ax.set_zlim(*zlim)
        ax.set_zlabel("elevation (m)")
        ax.set_title(label, fontsize=10)
        ax.set_xticks([]), ax.set_yticks([])
        fig.colorbar(pts, shrink=0.55, pad=0.08, label=cb_label)
    fig.suptitle(f"block {morton_decimal(pairs[pair][0])}", fontsize=11)
    plt.show()


_pair3d_dd = Dropdown(options=woptions, value=0, description="block")
_elev_cb = Checkbox(value=False, description="color by elevation (shared scale)")
_zmode_dd = Dropdown(
    options=[("independent z", "auto"), ("pin z to ATL03", "atl03"), ("pin z to GEDI", "gedi")],
    value="auto",
    description="z extent",
)
_out3d = interactive_output(paired_3d, {"pair": _pair3d_dd, "by_elev": _elev_cb, "zmode": _zmode_dd})
display(VBox([HBox([_pair3d_dd, _zmode_dd, _elev_cb]), _out3d]))

## Coincident waveforms

The cell-level join: one GEDI o18 cell against the 2×2 ATL03 o19 cells under
it, both reconstructed from their stored digests as Gaussian mixtures on a
shared elevation axis. The slider ranks joint cells by the *weaker* member
(`min(photons, pe)`), so early picks have real data on both sides. GEDI's
chunk grid puts one o12 block in one chunk; ATL03's chunks sit at o13.

In [5]:
%matplotlib inline
from ipywidgets import FloatText


def _mixture(digest, z, sigma):
    mu, wt = digest[:, 0], digest[:, 1]
    pdf = (wt[None, :] * np.exp(-0.5 * ((z[:, None] - mu[None, :]) / sigma) ** 2)).sum(axis=1)
    return pdf / max(wt.sum(), 1e-9) / (sigma * np.sqrt(2 * np.pi))


def paired_waveform(pair=0, nth=0, binw=1.0):
    w, joint, A2, G2 = pairs[pair]
    _, _, (aoff, ag), _ = ablocks[w]
    _, _, (goff, gg), _ = gblocks[w]
    rank = np.minimum(A2, G2) * joint
    order = np.argsort(rank.ravel())[::-1]
    r, c = np.unravel_index(int(order[min(nth, int(joint.sum()) - 1)]), rank.shape)

    gdigest = read_cell(gstore, GFIELD, cell_index(gstore, GFIELD, int(w), int(r), int(c)))
    kids = []
    for dr in (0, 1):
        for dc in (0, 1):
            rr, cc = 2 * r + dr, 2 * c + dc
            cid = int(generate_morton_children(int(w), 13)[rowcol_to_rank(rr // 64, cc // 64, depth=1)])
            try:
                kids.append(read_cell(astore, AFIELD, cell_index(astore, AFIELD, cid, rr % 64, cc % 64)))
            except Exception:
                pass
    adigest = np.concatenate([k for k in kids if len(k)]) if kids else np.empty((0, 2))

    lo = min(gdigest[:, 0].min(), adigest[:, 0].min()) - 5
    hi = max(gdigest[:, 0].max(), adigest[:, 0].max()) + 5
    z = np.linspace(lo, hi, 700)
    amu, awt = adigest[:, 0], adigest[:, 1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
    # bottom x-axis: the two digest-mixture densities (probability / m)
    ax1.plot(_mixture(gdigest, z, gg), z, color="#7b3294", lw=2, label=f"GEDI flux ({gdigest[:, 1].sum():.0f} pe)")
    ax1.plot(_mixture(adigest, z, ag), z, color="#008837", lw=2, label=f"ATL03 signal ({awt.sum():.0f} ph)")
    ax1.set_xlabel("normalized density")
    ax1.set_ylabel("elevation (m)")
    # top x-axis: RAW ATL03 photons -- binned bars at the chosen width, or the
    # individual photons as a column of dots when the width is 0 (SERC cells
    # are loss-free at delta=8192, so centroids ~ photons; merged centroids
    # carry their weight)
    ax1t = ax1.twiny()
    if binw and binw > 0:
        edges = np.arange(lo, hi + binw, binw)
        counts, _ = np.histogram(amu, bins=edges, weights=awt)
        ax1t.barh(edges[:-1] + binw / 2, counts, height=binw * 0.9,
                  color="#008837", alpha=0.25, zorder=0)
        ax1t.set_xlabel(f"ATL03 photons / {binw:g} m bin", fontsize=9)
    else:
        ax1t.scatter(np.zeros(len(amu)), amu, s=np.clip(awt * 8, 8, 40),
                     color="#008837", alpha=0.45, marker="o", zorder=0)
        ax1t.set_xlim(-0.05, 1.0)
        ax1t.set_xlabel("ATL03 photons (unbinned)", fontsize=9)
    ax1.set_zorder(ax1t.get_zorder() + 1)
    ax1.patch.set_visible(False)
    ax1.set_title(
        f"cell ({r},{c}) @o18 — GEDI {len(gdigest)} centroids vs ATL03 {len(adigest)} (2×2 @o19)",
        fontsize=9,
    )
    ax1.legend(fontsize=8)
    # cdf_from_tdigest returns CUMULATIVE WEIGHT (pe for GEDI, photon counts
    # for ATL03) -- normalize each by its total so both live in probability
    # space and share the axis honestly.
    ax2.plot(cdf_from_tdigest(gdigest, z) / max(gdigest[:, 1].sum(), 1e-9), z, color="#7b3294", lw=2)
    ax2.plot(cdf_from_tdigest(adigest, z) / max(adigest[:, 1].sum(), 1e-9), z, color="#008837", lw=2)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("CDF (probability)")
    ax2.set_title("cumulative", fontsize=10)
    for ax in (ax1, ax2):
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(alpha=0.25, lw=0.5)
    fig.suptitle(f"block {morton_decimal(w)}", fontsize=11)
    plt.tight_layout()
    plt.show()


_pairw_dd = Dropdown(options=woptions, value=0, description="block")
_nth_sl = IntSlider(min=0, max=40, value=0, description="nth joint")
_binw_ft = FloatText(value=1.0, step=0.5, description="bin (m)")
_outw = interactive_output(paired_waveform, {"pair": _pairw_dd, "nth": _nth_sl, "binw": _binw_ft})
display(VBox([HBox([_pairw_dd, _nth_sl, _binw_ft]), _outw]))

In [ ]:
(OUT / "timings_paired.json").write_text(json.dumps(timings, indent=2))
timings